# Unit 6: 高级CNN架构与迁移学习

## 学习目标
- 了解经典CNN架构的设计思想
- 掌握ResNet残差连接原理
- 学会使用torchvision预训练模型
- 掌握迁移学习的策略与实现
- 学会微调(Fine-tuning)技术
- 掌握数据增强技术

## 参考资源
- [torchvision.models文档](https://pytorch.org/vision/stable/models.html)
- [PyTorch教程 - Transfer Learning](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [ResNet论文](https://arxiv.org/abs/1512.03385)

## 6.1 经典CNN架构回顾

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import ResNet18_Weights, VGG16_Weights
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("6.1 经典CNN架构对比")
print("=" * 60)

architectures = {
    'LeNet-5 (1998)': {
        'params': '~60K',
        'layers': 5,
        'key_innovation': '首次成功应用CNN到手写数字识别',
    },
    'AlexNet (2012)': {
        'params': '~60M',
        'layers': 8,
        'key_innovation': 'ReLU激活、Dropout、数据增强、GPU训练',
    },
    'VGG-16 (2014)': {
        'params': '~138M',
        'layers': 16,
        'key_innovation': '使用小卷积核(3x3)堆叠增加深度',
    },
    'GoogLeNet (2014)': {
        'params': '~5M',
        'layers': 22,
        'key_innovation': 'Inception模块、1x1卷积降维',
    },
    'ResNet (2015)': {
        'params': '~25M (ResNet-50)',
        'layers': '50/101/152',
        'key_innovation': '残差连接、解决梯度消失问题',
    },
}

for name, info in architectures.items():
    print(f"\n{name}:")
    print(f"  参数量: {info['params']}")
    print(f"  层数: {info['layers']}")
    print(f"  核心创新: {info['key_innovation']}")

## 6.2 ResNet残差块实现

In [ ]:
print("=" * 60)
print("6.2 ResNet残差块实现")
print("=" * 60)

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = x
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out += self.shortcut(identity)
        out = F.relu(out)
        
        return out

block = BasicBlock(in_channels=64, out_channels=64, stride=1)
print(f"BasicBlock结构:\n{block}")

x = torch.randn(2, 64, 32, 32)
output = block(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"\n残差连接: output = F(x) + x")
print(f"核心思想: 学习残差映射F(x) = output - x，而非直接学习目标映射")

## 6.3 使用torchvision预训练模型

In [ ]:
print("=" * 60)
print("6.3 加载预训练模型")
print("=" * 60)

resnet18 = models.resnet18(weights=ResNet18_Weights.DEFAULT)
print(f"ResNet-18预训练模型:")
print(f"  总参数量: {sum(p.numel() for p in resnet18.parameters()):,}")
print(f"  分类类别数: {resnet18.fc.out_features}")

vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT)
print(f"\nVGG-16预训练模型:")
print(f"  总参数量: {sum(p.numel() for p in vgg16.parameters()):,}")
print(f"  分类类别数: {vgg16.classifier[-1].out_features}")

available_models = ['resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152',
                   'vgg11', 'vgg13', 'vgg16', 'vgg19',
                   'densenet121', 'densenet169',
                   'mobilenet_v2', 'efficientnet_b0']
print(f"\ntorchvision可用的预训练模型:")
for model_name in available_models:
    print(f"  - {model_name}")

## 6.4 迁移学习策略

In [ ]:
print("=" * 60)
print("6.4 迁移学习策略")
print("=" * 60)

print("迁移学习的三种常见策略:\n")

print("策略1: 特征提取器(冻结所有卷积层)")
print("  - 冻结预训练模型的所有卷积层")
print("  - 仅训练新添加的全连接层")
print("  - 适用场景: 数据集较小，与ImageNet相似\n")

print("策略2: 微调(解冻部分层)")
print("  - 冻结前面的卷积层")
print("  - 微调后面的卷积层和全连接层")
print("  - 适用场景: 数据集中等大小\n")

print("策略3: 完全微调(解冻所有层)")
print("  - 使用较小的学习率微调所有层")
print("  - 适用场景: 数据集较大，与ImageNet差异大\n")

print("选择建议:")
print("  数据集小 + 相似 → 策略1")
print("  数据集小 + 不相似 → 策略1或2")
print("  数据集大 + 相似 → 策略2")
print("  数据集大 + 不相似 → 策略3")

## 6.5 特征提取器实现

In [ ]:
print("=" * 60)
print("6.5 特征提取器实现")
print("=" * 60)

num_classes_new = 5

model_feature_extractor = models.resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model_feature_extractor.parameters():
    param.requires_grad = False

num_features = model_feature_extractor.fc.in_features
model_feature_extractor.fc = nn.Linear(num_features, num_classes_new)

trainable_params = sum(p.numel() for p in model_feature_extractor.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_feature_extractor.parameters())

print(f"特征提取器模式:")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")
print(f"  冻结参数量: {total_params - trainable_params:,}")
print(f"  可训练比例: {trainable_params/total_params*100:.2f}%")

x = torch.randn(2, 3, 224, 224)
output = model_feature_extractor(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

## 6.6 微调实现

In [ ]:
print("=" * 60)
print("6.6 微调实现")
print("=" * 60)

model_finetune = models.resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model_finetune.parameters():
    param.requires_grad = False

for param in model_finetune.layer4.parameters():
    param.requires_grad = True

num_features = model_finetune.fc.in_features
model_finetune.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes_new)
)

trainable_params = sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_finetune.parameters())

print(f"微调模式:")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")
print(f"  可训练比例: {trainable_params/total_params*100:.2f}%")

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_finetune.parameters()),
    lr=0.001
)
print(f"\n优化器仅更新可训练参数")
print(f"优化器管理的参数量: {sum(p.numel() for p in optimizer.param_groups[0]['params']):,}")

## 6.7 分层学习率

In [ ]:
print("=" * 60)
print("6.7 分层学习率设置")
print("=" * 60)

model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes_new)

optimizer_layered = torch.optim.Adam([
    {'params': model.conv1.parameters(), 'lr': 1e-5},
    {'params': model.layer1.parameters(), 'lr': 1e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-4},
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-3},
    {'params': model.fc.parameters(), 'lr': 1e-3},
])

print("分层学习率配置:")
for i, param_group in enumerate(optimizer_layered.param_groups):
    num_params = sum(p.numel() for p in param_group['params'])
    print(f"  参数组{i+1}: lr={param_group['lr']:.0e}, 参数量={num_params:,}")

## 6.8 数据增强技术

In [ ]:
print("=" * 60)
print("6.8 数据增强技术")
print("=" * 60)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])

print("训练集数据增强:")
print("  - RandomResizedCrop: 随机裁剪并缩放到224x224")
print("  - RandomHorizontalFlip: 50%概率水平翻转")
print("  - RandomRotation: 随机旋转±15度")
print("  - ColorJitter: 随机调整亮度、对比度、饱和度、色调")
print("  - RandomAffine: 随机平移10%")
print("  - Normalize: 使用ImageNet统计量标准化")

print("\n验证集数据增强:")
print("  - Resize: 缩放到256x256")
print("  - CenterCrop: 中心裁剪224x224")
print("  - Normalize: 使用ImageNet统计量标准化")

print("\n数据增强的作用:")
print("  1. 增加训练数据多样性")
print("  2. 提高模型泛化能力")
print("  3. 防止过拟合")
print("  4. 增强模型对变换的鲁棒性")

## 6.9 完整的迁移学习训练示例

In [ ]:
print("=" * 60)
print("6.9 迁移学习完整训练流程")
print("=" * 60)

from utils import get_device

def train_transfer_learning(model, train_loader, val_loader, num_epochs=10, 
                           learning_rate=0.001, device=None):
    if device is None:
        device = get_device()
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100.0 * val_correct / val_total
        val_loss = val_loss / val_total
        
        scheduler.step()
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_transfer_model.pth')
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    print(f"\n训练完成! 最佳验证准确率: {best_val_acc:.2f}%")
    return model

print("迁移学习训练函数已定义")
print("\n使用示例:")
print("  model = models.resnet18(weights=ResNet18_Weights.DEFAULT)")
print("  model.fc = nn.Linear(model.fc.in_features, num_classes)")
print("  model = train_transfer_learning(model, train_loader, val_loader)")

## 本章小结

本单元我们学习了：
1. 经典CNN架构的发展脉络
2. ResNet残差块的原理与实现
3. torchvision预训练模型的使用
4. 迁移学习的三种策略
5. 特征提取器模式的实现
6. 微调技术的实现
7. 分层学习率设置
8. 数据增强技术
9. 完整的迁移学习训练流程

## 练习建议
1. 比较不同预训练模型在相同任务上的表现
2. 实验不同的微调策略
3. 尝试不同的数据增强组合
4. 实现分层学习率并观察训练效果

## 下一步
进入Unit 7，完成综合实战项目，整合所有所学知识。